# 01 — Inspect the materialized Cosmos containers

After a sync run, browse the three target containers (`courses`, `learners`,
`recommendations`) and assert the documents have the expected denormalized
shape.

Auth: if `COSMOS_KEY` is set in `.env` we use it (emulator / local). Otherwise
we fall back to `DefaultAzureCredential` (managed identity in Azure, az login
locally).


In [ ]:
from pathlib import Path
import os

from dotenv import load_dotenv
from azure.cosmos import CosmosClient

repo_root = Path.cwd()
while not (repo_root / ".env.example").exists():
    if repo_root == repo_root.parent:
        raise RuntimeError("could not find repo root (.env.example missing)")
    repo_root = repo_root.parent

load_dotenv(repo_root / ".env")

endpoint = os.environ["COSMOS_ENDPOINT"]
key = os.environ.get("COSMOS_KEY")  # emulator / local
db_name = os.environ.get("COSMOS_DATABASE", "learnsphere")

if key:
    client = CosmosClient(endpoint, credential=key)
else:
    from azure.identity import DefaultAzureCredential
    client = CosmosClient(endpoint, credential=DefaultAzureCredential())

db = client.get_database_client(db_name)
print(f"endpoint={endpoint}  database={db_name}  auth={'key' if key else 'aad'}")


## Sample a course document


In [ ]:
container = db.get_container_client("courses")
items = list(container.query_items(
    "SELECT TOP 1 * FROM c", enable_cross_partition_query=True
))
items[0] if items else "empty container"


## Sample a learner document — note the embedded enrollment summary


In [ ]:
container = db.get_container_client("learners")
items = list(container.query_items(
    "SELECT TOP 1 * FROM c WHERE c.enrollmentStats.totalEnrolled > 0",
    enable_cross_partition_query=True,
))
items[0] if items else "no learner with enrollments yet"


## Sample a recommendations document


In [ ]:
container = db.get_container_client("recommendations")
items = list(container.query_items(
    "SELECT TOP 1 * FROM c", enable_cross_partition_query=True
))
items[0] if items else "empty container"
